# Personalization-Placement Ablation — Analysis (post-split judges, 6-variant grid)

**The analysis notebook for this experiment — built and ready, waiting on data.** Every section is wired
up; it's empty right now only because the post-split re-run/re-score hasn't been done yet. Once it has,
**set `RESULTS_DIR` below and Run-All** and every section fills in automatically. It loads only post-split
results and never mixes in old / pre-split / relabeled runs.

### What "post-split" means here
The final-response judge is now **two** judges, merged into one `scores` dict:
- **Answer-Quality** (rubric + answer, *blind to evidence*): `intent_satisfaction`, `personalization_target_use`,
  `overpersonalization`, `specificity`, `safety`, `non_genericness`, `domain_safety`,
  `missing_constraint_awareness`, `actionability_without_overclaiming`, `uncertainty_calibration`.
- **Evidence-Faithfulness** (evidence + answer, *blind to persona/rubric*): `groundedness`,
  `unsupported_claim_risk`, `contradiction_with_evidence`, `citation_support`, `evidence_usage_quality`.
- Plus the **Retrieval** judge (7) and **Fan-out** judge (6) score files.

### The 6-variant grid
| | generic synthesis | personalized synthesis |
|---|---|---|
| **generic fan-out** | V1 | V2 |
| **personalized fan-out** | **V3** fanout_only | **V4** personalized_fanout |

(+ V0 single baseline, V5 mixed.) After the re-run all six exist, so the cleanest tests — `V3−V1`
(fan-out-only) and the fan-out×synthesis **interaction** — become computable.

### How to populate
1. Re-run the agent for all 6 variants and re-score with the current split judges (writing to `RESULTS_DIR`).
2. Set `RESULTS_DIR` / `RUNS_PATH` in the next cell, **Run All**.

### Coverage (this edition)
The 2×2 placement grid (§2) and the marginal effects + fan-out×synthesis interaction (§3) are the headline
figures. This edition also covers: the full final-metric sweep (§1), the retrieval-stage (§4) and
fan-out-stage (§5) judges, task-type flip (§6) and per-domain (§7) generalization, the over-personalization
trade-off (§8), quality-vs-faithfulness (§9), cost (§10), and appendix summary tables + CSVs (§11). Every
figure exports to `RESULTS_DIR/figures/` (PDF+PNG) on Run-All.

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"]=(10,6); plt.rcParams["axes.edgecolor"]="#cccccc"; plt.rcParams["axes.linewidth"]=0.8
np.random.seed(0)

PROJECT_ROOT=os.path.dirname(os.getcwd()); OUT=os.path.join(PROJECT_ROOT,"outputs")
# ----------------------------------------------------------------------------------------------------
# SET THIS to the directory your post-split re-run/re-score wrote to. It must contain:
#   final_response_scores.jsonl, retrieval_scores.jsonl, fanout_scores.jsonl, and (for macro_domain) runs.jsonl
RESULTS_DIR = os.path.join(OUT, "placement_ablation_v1_postsplit")   # <-- does not exist yet; create via re-run
RUNS_PATH   = os.path.join(RESULTS_DIR, "runs.jsonl")
# ----------------------------------------------------------------------------------------------------

VARIANTS=["V0_generic_single","V1_generic_fanout","V2_synthesis_only_personalization",
          "V3_fanout_only_personalization","V4_personalized_fanout","V5_mixed_fanout"]
VLABEL={v:v.split("_")[0] for v in VARIANTS}; SHORT=[VLABEL[v] for v in VARIANTS]; VKEY={k.split("_")[0]:k for k in VARIANTS}
VCOLORS={"V0_generic_single":"#cccccc","V1_generic_fanout":"#999999",
         "V2_synthesis_only_personalization":"#4477aa","V3_fanout_only_personalization":"#ee8866",
         "V4_personalized_fanout":"#44bb99","V5_mixed_fanout":"#aa4499"}

QUALITY_METRICS=["intent_satisfaction","personalization_target_use","overpersonalization","specificity","safety",
                 "non_genericness","domain_safety","missing_constraint_awareness","actionability_without_overclaiming","uncertainty_calibration"]
FAITH_METRICS=["groundedness","unsupported_claim_risk","contradiction_with_evidence","citation_support","evidence_usage_quality"]
RETR_METRICS=["evidence_relevance","result_persona_fit","constraint_coverage","distractor_robustness","source_quality","disconfirming_coverage","unsafe_or_overpersonalized_retrieval_risk"]
FANOUT_METRICS=["persona_field_use","query_specificity","query_diversity","search_realism","faithfulness_to_user_query","overpersonalization_risk"]
LOWER_IS_BETTER={"overpersonalization","unsupported_claim_risk","contradiction_with_evidence",
                 "unsafe_or_overpersonalized_retrieval_risk","overpersonalization_risk"}

def _read(name, keys):
    p=os.path.join(RESULTS_DIR,name); out={}
    if not os.path.exists(p): return out
    for l in open(p):
        if not l.strip(): continue
        d=json.loads(l); s=d.get("scores") or {}
        if not s: continue
        out[d["run_id"]]=dict(variant=d.get("variant"),task_type=d.get("task_type"),query_id=d.get("query_id"),
                              **{k:s.get(k) for k in keys})
    return out

def load():
    fin=_read("final_response_scores.jsonl", QUALITY_METRICS+FAITH_METRICS)
    if not fin: return pd.DataFrame()
    retr=_read("retrieval_scores.jsonl", RETR_METRICS); fan=_read("fanout_scores.jsonl", FANOUT_METRICS)
    meta={}
    if os.path.exists(RUNS_PATH):
        for l in open(RUNS_PATH):
            if l.strip(): r=json.loads(l); meta[r["run_id"]]=r.get("macro_domain")
    rows=[]
    for rid,f in fin.items():
        rows.append(dict(run_id=rid, macro_domain=meta.get(rid),
                         **f,
                         **{k:retr.get(rid,{}).get(k) for k in RETR_METRICS},
                         **{k:fan.get(rid,{}).get(k) for k in FANOUT_METRICS}))
    df=pd.DataFrame(rows); df["v"]=df["variant"].map(VLABEL); return df

DF=load(); HAS_DATA=not DF.empty
if HAS_DATA:
    print("Loaded post-split scores:", len(DF), "rows")
    print("variants present:", [VLABEL[v] for v in VARIANTS if v in set(DF.variant)],
          "| missing:", [VLABEL[v] for v in VARIANTS if v not in set(DF.variant)])
else:
    print(f"⏳ No post-split scores found in {RESULTS_DIR}.")
    print("   Run the 6-variant re-run + re-score (current split judges) into that dir, then Run-All.")

WAIT="⏳ No post-split data yet — populate RESULTS_DIR (re-run + re-score), then re-run this cell."

def paired(a, b, metric, tt=None, B=5000, seed=0):
    """Paired per-query Δ(a-b) ± bootstrap 95% CI. Per-call seed => order-independent.
    status: ok | pending (variant absent) | empty (no overlapping non-NaN queries)."""
    if DF.empty: return dict(mean=np.nan,lo=np.nan,hi=np.nan,n=0,status="nodata")
    sub=DF if tt is None else DF[DF.task_type==tt]; va,vb=VKEY[a],VKEY[b]
    if va not in set(sub.variant) or vb not in set(sub.variant): return dict(mean=np.nan,lo=np.nan,hi=np.nan,n=0,status="pending")
    pa=sub[sub.variant==va].groupby("query_id")[metric].mean(); pb=sub[sub.variant==vb].groupby("query_id")[metric].mean()
    common=pa.index.intersection(pb.index); d=(pa.loc[common]-pb.loc[common]).values; d=d[~np.isnan(d)]; n=len(d)
    if n==0: return dict(mean=np.nan,lo=np.nan,hi=np.nan,n=0,status="empty")
    rng=np.random.default_rng(seed); boot=np.array([rng.choice(d,n,replace=True).mean() for _ in range(B)])
    return dict(mean=float(d.mean()),lo=float(np.percentile(boot,2.5)),hi=float(np.percentile(boot,97.5)),n=n,status="ok")

# ---- extra helpers for the full-coverage analysis (build on the objects above) ----
FIGDIR = os.path.join(RESULTS_DIR, "figures")

def save_fig(fig, name):
    """Export a figure to RESULTS_DIR/figures as PDF (paper) + PNG (preview)."""
    try:
        os.makedirs(FIGDIR, exist_ok=True)
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"), bbox_inches="tight", dpi=200)
    except Exception as e:
        print("save_fig skipped:", e)

def vmean_ci(metric, variants=VARIANTS, tt=None, dom=None, B=2000, seed=0):
    """Per-variant mean + bootstrap 95% CI. Returns {variant: (mean, lo, hi, n)}."""
    sub = DF
    if tt is not None: sub = sub[sub.task_type == tt]
    if dom is not None: sub = sub[sub.macro_domain == dom]
    rng = np.random.default_rng(seed); out = {}
    for v in variants:
        x = sub[sub.variant == v][metric].dropna().values if metric in sub else np.array([])
        if len(x) == 0: out[v] = (np.nan, np.nan, np.nan, 0); continue
        boot = np.array([rng.choice(x, len(x), replace=True).mean() for _ in range(B)])
        out[v] = (float(x.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)), len(x))
    return out

def bars_ci(ax, metric, tt=None, title=None):
    """Per-variant mean bars with bootstrap 95% CI onto ax. Skips absent metric."""
    if metric not in DF:
        ax.set_title(f"{metric} (absent)"); ax.set_axis_off(); return
    ci = vmean_ci(metric, tt=tt); m = [ci[v][0] for v in VARIANTS]
    lo = [0 if np.isnan(ci[v][0]) else ci[v][0]-ci[v][1] for v in VARIANTS]
    hi = [0 if np.isnan(ci[v][0]) else ci[v][2]-ci[v][0] for v in VARIANTS]
    ax.bar(SHORT, [0 if np.isnan(x) else x for x in m], yerr=[lo, hi],
           color=[VCOLORS[v] for v in VARIANTS], edgecolor="#555", capsize=3, error_kw=dict(elinewidth=1))
    ax.set_ylim(0, 5.5); ax.set_ylabel("mean (1-5)")
    ax.set_title((title or metric.replace("_", " ").title()) + ("  (lower=better)" if metric in LOWER_IS_BETTER else ""),
                 fontsize=11, fontweight="bold")

def interaction(metric, tt=None, B=5000, seed=0):
    """Bootstrap the fan-out x synthesis interaction (V4-V2)-(V3-V1) over per-query double-diffs."""
    if DF.empty or metric not in DF: return dict(mean=np.nan, lo=np.nan, hi=np.nan, n=0, status="nodata")
    sub = DF if tt is None else DF[DF.task_type == tt]
    piv = sub.pivot_table(index="query_id", columns="variant", values=metric, aggfunc="mean")
    need = [VKEY[k] for k in ("V1", "V2", "V3", "V4")]
    if any(c not in piv.columns for c in need): return dict(mean=np.nan, lo=np.nan, hi=np.nan, n=0, status="pending")
    d = ((piv[VKEY["V4"]]-piv[VKEY["V2"]]) - (piv[VKEY["V3"]]-piv[VKEY["V1"]])).dropna().values; n = len(d)
    if n == 0: return dict(mean=np.nan, lo=np.nan, hi=np.nan, n=0, status="empty")
    rng = np.random.default_rng(seed); boot = np.array([rng.choice(d, n, replace=True).mean() for _ in range(B)])
    return dict(mean=float(d.mean()), lo=float(np.percentile(boot, 2.5)), hi=float(np.percentile(boot, 97.5)), n=n, status="ok")

# cost_proxy per run (optional; only if runs.jsonl present in RESULTS_DIR)
COST = pd.DataFrame()
if HAS_DATA and os.path.exists(RUNS_PATH):
    _cr = []
    for _l in open(RUNS_PATH):
        if not _l.strip(): continue
        _r = json.loads(_l); _cp = _r.get("cost_proxy") or {}
        _cr.append(dict(run_id=_r.get("run_id"), variant=_r.get("variant"),
                        **{k: _cp.get(k) for k in ("num_gemini_calls", "num_tavily_calls", "num_fanout_branches", "num_raw_results")}))
    COST = pd.DataFrame(_cr)


## Section 0 — Data coverage / sanity

Confirms what loaded, n per (variant × task-type), and any errored rows. (Populates after the re-run.)

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    print("n per (variant, task_type):")
    print(DF.groupby(["v","task_type"]).size().unstack(fill_value=0).reindex(SHORT))
    for fam,ms in [("Quality",QUALITY_METRICS),("Faithfulness",FAITH_METRICS),("Retrieval",RETR_METRICS),("Fan-out",FANOUT_METRICS)]:
        miss=[m for m in ms if m in DF and DF[m].isna().all()]
        print(f"  {fam}: {len(ms)} metrics" + (f"  [MISSING: {miss}]" if miss else ""))

## Section 1 — Overall performance by variant (full metric sweep)

Every final-answer metric at a glance: a variant × metric heatmap (Answer-Quality + Evidence-Faithfulness),
then per-variant bars with bootstrap 95% CI for the headline metrics. This is the appendix-style overview;
the curated story is §2–§3.

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    fin_metrics = [m for m in QUALITY_METRICS + FAITH_METRICS if m in DF] or ["intent_satisfaction"]
    H = DF.groupby("variant")[fin_metrics].mean().reindex(VARIANTS); H.index = SHORT
    fig, ax = plt.subplots(figsize=(min(2.0 + 0.55*len(fin_metrics), 13), 4.4))
    sns.heatmap(H, annot=True, fmt=".2f", cmap="viridis", vmin=1, vmax=5, linewidths=.5, linecolor="white",
                cbar_kws={"label": "mean (1-5)"}, annot_kws={"fontsize": 7}, ax=ax)
    ax.set_title("All final-answer metrics by variant (mean; lower-is-better cols not reoriented)", fontsize=11, fontweight="bold")
    ax.set_xticklabels([t.get_text().replace("_", "\n") for t in ax.get_xticklabels()], fontsize=6, rotation=90)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    plt.tight_layout(); save_fig(fig, "all_final_metrics_heatmap"); plt.show()

    head = [m for m in ["intent_satisfaction","personalization_target_use","result_persona_fit","constraint_coverage","overpersonalization","domain_safety"] if m in DF]
    fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.flatten()
    for i, m in enumerate(head): bars_ci(axes[i], m)
    for j in range(len(head), len(axes)): axes[j].set_axis_off()
    plt.suptitle("Headline metrics by variant (bootstrap 95% CI; bars from 0)", y=1.02, fontsize=12, fontweight="bold")
    plt.tight_layout(); save_fig(fig, "headline_bars"); plt.show()

## Section 2 — The 2×2 placement grid

Down a column = fan-out personalization; along a row = synthesis personalization. The four cells are
V1 / V2 / V3 (fan-out-only) / V4 (personalized fan-out). Shown for the key placement metrics.

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    def grid(metric):
        m = DF.groupby("variant")[metric].mean()
        M = pd.DataFrame(index=["generic fan-out", "personalized fan-out"],
                         columns=["generic synthesis", "personalized synthesis"], dtype=float)
        M.iloc[0, 0] = m.get(VKEY["V1"], np.nan); M.iloc[0, 1] = m.get(VKEY["V2"], np.nan)
        M.iloc[1, 0] = m.get(VKEY["V3"], np.nan); M.iloc[1, 1] = m.get(VKEY["V4"], np.nan)
        return M
    hm = [m for m in ["intent_satisfaction","personalization_target_use","result_persona_fit","constraint_coverage"] if m in DF] or ["intent_satisfaction"]
    fig, axes = plt.subplots(1, len(hm), figsize=(5.2*len(hm), 5)); axes = np.atleast_1d(axes)
    for ax, metric in zip(axes, hm):
        sns.heatmap(grid(metric), annot=True, fmt=".2f", cmap="viridis", vmin=1, vmax=5, linewidths=2, linecolor="white",
                    cbar_kws={"label": "mean (1-5)"}, annot_kws={"fontsize": 13, "fontweight": "bold"}, ax=ax)
        ax.set_title(metric.replace("_", " ").title(), fontsize=12, fontweight="bold")
        ax.set_xlabel("→ synthesis personalization"); ax.set_ylabel("↓ fan-out personalization")
    plt.suptitle("2×2 placement grid: fan-out (rows) vs synthesis (cols)", y=1.05, fontsize=12, fontweight="bold")
    plt.tight_layout(); save_fig(fig, "placement_grid"); plt.show()

## Section 3 — Marginal effects + the fan-out×synthesis interaction

Left: paired Δ ± bootstrap 95% CI for every placement contrast, per headline metric. Right: the interaction
(V4−V2)−(V3−V1) with its **own** bootstrap CI, per metric — >0 means fan-out personalization helps MORE when
synthesis is also personalized (super-additive). CIs uncorrected for multiple comparisons → directional.

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    METS = [m for m in ["intent_satisfaction","personalization_target_use","result_persona_fit","constraint_coverage"] if m in DF] or ["intent_satisfaction"]
    CONTR = [("V2","V1","synth | generic fanout"), ("V4","V3","synth | pers fanout"),
             ("V3","V1","fanout | generic synth"), ("V4","V2","fanout | pers synth"),
             ("V4","V1","full personalization"), ("V5","V4","mixed vs pers fanout")]
    fig, axes = plt.subplots(1, len(METS), figsize=(5.0*len(METS), 5.0), sharey=True); axes = np.atleast_1d(axes)
    for ax, metric in zip(axes, METS):
        for yi, (a, b, nm) in enumerate(CONTR):
            r = paired(a, b, metric)
            if r["status"] != "ok":
                ax.text(0.0, yi, r["status"], va="center", ha="center", color="#cc3333", fontsize=8)
            else:
                ax.errorbar(r["mean"], yi, xerr=[[r["mean"]-r["lo"]], [r["hi"]-r["mean"]]], fmt="o", color="#4477aa", capsize=3, ms=6)
        ax.axvline(0, color="#888", ls="--"); ax.set_yticks(range(len(CONTR)))
        ax.set_yticklabels([f"{a}-{b} {nm}" for a, b, nm in CONTR], fontsize=8); ax.invert_yaxis()
        ax.set_title(metric.replace("_", " ").title(), fontsize=11, fontweight="bold"); ax.set_xlabel("paired Δ")
    plt.suptitle("Marginal placement effects (paired Δ ± 95% CI)", y=1.03, fontsize=12, fontweight="bold")
    plt.tight_layout(); save_fig(fig, "marginal_effects"); plt.show()

    fig, ax = plt.subplots(figsize=(8, 0.7*len(METS) + 1.6))
    for yi, metric in enumerate(METS):
        r = interaction(metric)
        if r["status"] != "ok":
            ax.text(0.0, yi, r["status"], va="center", ha="center", color="#cc3333", fontsize=9)
        else:
            ax.errorbar(r["mean"], yi, xerr=[[r["mean"]-r["lo"]], [r["hi"]-r["mean"]]], fmt="s", color="#aa4499", capsize=4, ms=8)
            ax.text(r["hi"] + 0.02, yi, f"{r['mean']:+.2f}", va="center", fontsize=9)
    ax.axvline(0, color="#888", ls="--"); ax.set_yticks(range(len(METS)))
    ax.set_yticklabels([m.replace("_", " ").title() for m in METS]); ax.invert_yaxis()
    ax.set_title("Fan-out × synthesis INTERACTION (V4-V2)-(V3-V1), bootstrap 95% CI", fontsize=11, fontweight="bold")
    ax.set_xlabel("interaction effect"); plt.tight_layout(); save_fig(fig, "interaction"); plt.show()

## Section 4 — Retrieval-stage effects (does personalized fan-out retrieve better evidence?)

The Retrieval judge (7 metrics) by variant — the retrieval mechanism this study rests on. Personalized fan-out (V3/V4) should
lift constraint_coverage / evidence_relevance / result_persona_fit and control
`unsafe_or_overpersonalized_retrieval_risk` (lower-is-better).

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    rm = [m for m in RETR_METRICS if m in DF]
    if not rm:
        print("no retrieval metrics loaded.")
    else:
        fig, axes = plt.subplots(2, 4, figsize=(20, 9)); axes = axes.flatten()
        for i, m in enumerate(rm): bars_ci(axes[i], m)
        for j in range(len(rm), len(axes)): axes[j].set_axis_off()
        plt.suptitle("Retrieval judge metrics by variant (bootstrap 95% CI)", y=1.02, fontsize=12, fontweight="bold")
        plt.tight_layout(); save_fig(fig, "retrieval_by_variant"); plt.show()

## Section 5 — Fan-out-stage effects (how does personalizing the fan-out change the queries?)

The Fan-out judge (6 metrics) by variant — the direct read on the fan-out lever: does personalized fan-out
raise persona_field_use / diversity without inflating `overpersonalization_risk` (lower-is-better)?
Generic-fan-out variants may score N/A on persona fields (blank bars by design).

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    fm = [m for m in FANOUT_METRICS if m in DF]
    if not fm:
        print("no fan-out metrics loaded.")
    else:
        fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.flatten()
        for i, m in enumerate(fm): bars_ci(axes[i], m)
        for j in range(len(fm), len(axes)): axes[j].set_axis_off()
        plt.suptitle("Fan-out judge metrics by variant (bootstrap 95% CI)", y=1.02, fontsize=12, fontweight="bold")
        plt.tight_layout(); save_fig(fig, "fanout_by_variant"); plt.show()

## Section 6 — Task-type sensitivity (the flip)

The study's central claim: the best placement flips between retrieval- and synthesis-sensitive tasks. Each lever's
paired Δ (fan-out V3−V1, synthesis V2−V1, both V4−V1) split by task type, across the headline metrics.

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    effects = [("V3","V1","fan-out\n(V3-V1)"), ("V2","V1","synthesis\n(V2-V1)"), ("V4","V1","both\n(V4-V1)")]
    METS = [m for m in ["intent_satisfaction","result_persona_fit","constraint_coverage","groundedness"] if m in DF] or ["intent_satisfaction"]
    tts = ["retrieval_sensitive", "synthesis_sensitive"]; x = np.arange(len(effects)); w = 0.35
    fig, axes = plt.subplots(1, len(METS), figsize=(5.0*len(METS), 5)); axes = np.atleast_1d(axes)
    for ax, metric in zip(axes, METS):
        for k, tt in enumerate(tts):
            rs = [paired(a, b, metric, tt=tt) for a, b, _ in effects]
            m = [r["mean"] for r in rs]
            lo = [(r["mean"]-r["lo"]) if r["status"] == "ok" else 0 for r in rs]
            hi = [(r["hi"]-r["mean"]) if r["status"] == "ok" else 0 for r in rs]
            ax.bar(x + (k-0.5)*w, [0 if pd.isna(v) else v for v in m], w, yerr=[lo, hi], capsize=3,
                   label=tt, color=["#4477aa", "#ee8866"][k], edgecolor="#555", error_kw=dict(elinewidth=1))
        ax.axhline(0, color="#888", ls="--"); ax.set_xticks(x); ax.set_xticklabels([e[2] for e in effects], fontsize=8)
        ax.set_title(metric.replace("_", " ").title(), fontsize=11, fontweight="bold"); ax.set_ylabel("paired Δ")
    axes[0].legend(fontsize=8)
    plt.suptitle("Lever effects by task type (paired Δ ± 95% CI)", y=1.03, fontsize=12, fontweight="bold")
    plt.tight_layout(); save_fig(fig, "tasktype_flip"); plt.show()

## Section 7 — Per macro-domain (education / legal / finance)

Key metrics by variant within each domain (bootstrap 95% CI; needs `runs.jsonl` for the domain join).
`domain_safety` and `constraint_coverage` matter most in high-stakes legal/finance.

In [ ]:
if not HAS_DATA: print(WAIT)
elif DF["macro_domain"].isna().all(): print("macro_domain unavailable — put runs.jsonl in RESULTS_DIR.")
else:
    doms = sorted(DF[DF.macro_domain.notna()].macro_domain.unique())
    METS = [m for m in ["intent_satisfaction", "constraint_coverage", "domain_safety"] if m in DF] or ["intent_satisfaction"]
    x = np.arange(len(doms)); w = 0.14
    fig, axes = plt.subplots(1, len(METS), figsize=(6.2*len(METS), 5)); axes = np.atleast_1d(axes)
    for ax, metric in zip(axes, METS):
        for i, v in enumerate(VARIANTS):
            ci = [vmean_ci(metric, variants=[v], dom=d)[v] for d in doms]
            means = [c[0] for c in ci]
            lo = [0 if np.isnan(c[0]) else c[0]-c[1] for c in ci]
            hi = [0 if np.isnan(c[0]) else c[2]-c[0] for c in ci]
            ax.bar(x + (i-2.5)*w, [0 if np.isnan(mm) else mm for mm in means], w, yerr=[lo, hi],
                   label=VLABEL[v], color=VCOLORS[v], edgecolor="#555", linewidth=0.4, capsize=2, error_kw=dict(elinewidth=0.6))
        ax.set_xticks(x); ax.set_xticklabels(doms, fontsize=9); ax.set_ylim(0, 5.5)
        ax.set_title(metric.replace("_", " ").title(), fontsize=11, fontweight="bold")
    axes[0].set_ylabel("mean (1-5)"); axes[-1].legend(title="variant", ncol=2, fontsize=7)
    plt.suptitle("Key metrics by macro-domain and variant (bootstrap 95% CI)", y=1.03, fontsize=12, fontweight="bold")
    plt.tight_layout(); save_fig(fig, "by_domain"); plt.show()

## Section 8 — The over-personalization trade-off

Personalizing fan-out can narrow retrieval and miss disconfirming / constraint evidence. Left: the trade-off
scatter — personalization gain (`personalization_target_use`) vs `disconfirming_coverage`, per variant
(bottom-right = personalized but blinkered). Right: the over-personalization risk metrics by variant
(`overpersonalization`, `unsafe_or_overpersonalized_retrieval_risk`: lower-is-better).

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    fig, axes = plt.subplots(1, 2, figsize=(17, 6))
    ax = axes[0]; xm = "personalization_target_use"; ym = "disconfirming_coverage"
    if xm in DF and ym in DF:
        gx = DF.groupby("variant")[xm].mean(); gy = DF.groupby("variant")[ym].mean()
        for v in VARIANTS:
            if pd.notna(gx.get(v)) and pd.notna(gy.get(v)):
                ax.scatter(gx[v], gy[v], s=150, color=VCOLORS[v], edgecolor="#333", zorder=3)
                ax.annotate(VLABEL[v], (gx[v], gy[v]), xytext=(5, 5), textcoords="offset points", fontsize=10, fontweight="bold")
        ax.set_xlabel("personalization_target_use (gain)"); ax.set_ylabel("disconfirming_coverage (breadth kept)")
        ax.set_title("Trade-off: personalization vs disconfirming coverage", fontsize=11, fontweight="bold")
        ax.set_xlim(1, 5.2); ax.set_ylim(1, 5.2)
    else:
        ax.set_title("trade-off metrics absent"); ax.set_axis_off()
    ax = axes[1]
    risk = [m for m in ["overpersonalization","disconfirming_coverage","unsafe_or_overpersonalized_retrieval_risk","missing_constraint_awareness","overpersonalization_risk"] if m in DF]
    xr = np.arange(len(risk)); w = 0.13
    for i, v in enumerate(VARIANTS):
        vals = [DF[DF.variant == v][m].mean() for m in risk]
        ax.bar(xr + (i-2.5)*w, [0 if pd.isna(z) else z for z in vals], w, label=VLABEL[v], color=VCOLORS[v], edgecolor="#555", linewidth=0.4)
    ax.set_xticks(xr); ax.set_xticklabels([m.replace("_", "\n") for m in risk], fontsize=7); ax.set_ylim(0, 5.3)
    ax.set_title("Over-personalization risk metrics by variant (some lower=better)", fontsize=11, fontweight="bold"); ax.legend(title="variant", ncol=3, fontsize=7)
    plt.suptitle("Over-personalization trade-off", y=1.02, fontsize=12, fontweight="bold")
    plt.tight_layout(); save_fig(fig, "overpersonalization_tradeoff"); plt.show()

## Section 9 — Answer-Quality vs Evidence-Faithfulness (the split's payoff)

Does personalization raise answer quality without costing faithfulness? Left: quality
(`personalization_target_use`) vs faithfulness (`groundedness`) per variant. Right: the faithfulness
sub-metrics by variant (`unsupported_claim` / `contradiction`: lower=better).

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    ax = axes[0]
    qx = DF.groupby("variant")["personalization_target_use"].mean().reindex(VARIANTS)
    fy = DF.groupby("variant")["groundedness"].mean().reindex(VARIANTS)
    for v in VARIANTS:
        if pd.notna(qx[v]) and pd.notna(fy[v]):
            ax.scatter(qx[v], fy[v], s=140, color=VCOLORS[v], edgecolor="#333", zorder=3)
            ax.annotate(VLABEL[v], (qx[v], fy[v]), xytext=(5, 5), textcoords="offset points", fontsize=10, fontweight="bold")
    ax.set_xlabel("Answer-Quality: personalization_target_use"); ax.set_ylabel("Faithfulness: groundedness")
    ax.set_title("Quality vs Faithfulness per variant\n(top-right = personalized AND grounded)", fontsize=11, fontweight="bold")
    ax.set_xlim(1, 5.2); ax.set_ylim(1, 5.2)
    ax = axes[1]
    fsub = [m for m in ["groundedness","citation_support","evidence_usage_quality","unsupported_claim_risk","contradiction_with_evidence"] if m in DF]
    xf = np.arange(len(fsub)); w = 0.13
    for i, v in enumerate(VARIANTS):
        vals = DF[DF.variant == v][fsub].mean().reindex(fsub)
        ax.bar(xf + (i-2.5)*w, [0 if pd.isna(z) else z for z in vals.values], w, label=VLABEL[v], color=VCOLORS[v], edgecolor="#555", linewidth=0.4)
    ax.set_xticks(xf); ax.set_xticklabels([m.replace("_", "\n") for m in fsub], fontsize=8); ax.set_ylim(0, 5.3)
    ax.set_title("Evidence-Faithfulness sub-metrics by variant\n(unsupported_claim / contradiction: lower=better)", fontsize=11, fontweight="bold")
    ax.legend(title="variant", ncol=6, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    plt.tight_layout(); save_fig(fig, "quality_vs_faithfulness"); plt.show()

## Section 10 — Cost / efficiency of each placement

The levers differ in cost: personalized fan-out adds a planner call and more branches/searches. Per-variant
`cost_proxy` (from `runs.jsonl`) — read alongside the quality gains to weigh each placement.

In [ ]:
if not HAS_DATA: print(WAIT)
elif COST.empty: print("cost_proxy unavailable — put runs.jsonl in RESULTS_DIR.")
else:
    cm = [c for c in ["num_gemini_calls","num_tavily_calls","num_fanout_branches","num_raw_results"] if c in COST]
    g = COST.groupby("variant")[cm].mean().reindex(VARIANTS); g.index = SHORT
    fig, axes = plt.subplots(1, len(cm), figsize=(4.2*len(cm), 4.2)); axes = np.atleast_1d(axes)
    for ax, c in zip(axes, cm):
        ax.bar(SHORT, g[c].values, color=[VCOLORS[v] for v in VARIANTS], edgecolor="#555")
        ax.set_title(c.replace("num_", "").replace("_", " "), fontsize=10, fontweight="bold"); ax.tick_params(axis="x", labelsize=8)
    plt.suptitle("Mean cost proxy by variant", y=1.03, fontsize=12, fontweight="bold")
    plt.tight_layout(); save_fig(fig, "cost_by_variant"); plt.show()

## Section 11 — Summary tables (numbers behind the figures)

Per-variant mean for every metric, and the key paired contrasts with 95% CI — the appendix tables. Also
written to `RESULTS_DIR` as CSV for the paper.

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    allm = [m for m in QUALITY_METRICS + FAITH_METRICS + RETR_METRICS + FANOUT_METRICS if m in DF]
    tbl = DF.groupby("variant")[allm].mean().reindex(VARIANTS); tbl.index = SHORT
    try:
        from IPython.display import display; display(tbl.round(2))
    except Exception:
        print(tbl.round(2))
    CONTR = [("V2","V1"), ("V3","V1"), ("V4","V2"), ("V4","V1"), ("V5","V4")]
    rows = []
    for metric in [m for m in ["intent_satisfaction","personalization_target_use","result_persona_fit","constraint_coverage","groundedness","overpersonalization"] if m in DF]:
        for a, b in CONTR:
            r = paired(a, b, metric)
            rows.append(dict(metric=metric, contrast=f"{a}-{b}", delta=r["mean"], lo=r["lo"], hi=r["hi"], n=r["n"], status=r["status"]))
    ctab = pd.DataFrame(rows)
    try:
        from IPython.display import display; display(ctab.round(3))
    except Exception:
        print(ctab.round(3))
    try:
        os.makedirs(RESULTS_DIR, exist_ok=True)
        tbl.round(3).to_csv(os.path.join(RESULTS_DIR, "summary_by_variant_ALL.csv"))
        ctab.round(4).to_csv(os.path.join(RESULTS_DIR, "contrasts_ALL.csv"), index=False)
        print("wrote summary_by_variant_ALL.csv, contrasts_ALL.csv to", RESULTS_DIR)
    except Exception as e:
        print("csv write skipped:", e)

## Section 12 — How to read this & caveats

- **Section guide:** §1 full metric sweep · §2 placement grid · §3 marginal effects + interaction · §4 retrieval-stage · §5 fan-out-stage · §6 task-type flip · §7 per-domain · §8 over-personalization trade-off · §9 quality vs faithfulness · §10 cost · §11 summary tables.
- **Decoupled axes:** Answer-Quality (rubric, evidence-blind) and Evidence-Faithfulness (evidence, persona-blind) are separate, so personalization gains and any faithfulness cost are read independently.
- **Lower-is-better** metrics: `overpersonalization`, `unsupported_claim_risk`, `contradiction_with_evidence`, `unsafe_or_overpersonalized_retrieval_risk`, `overpersonalization_risk` (heatmaps are NOT reoriented — read those columns/bars accordingly).
- **`overpersonalization` is overloaded** — it fires on `should_not_use` violations, which generic variants commit by *under*-personalizing; read it next to the variant.
- **Statistics:** all CIs are bootstrap (2–5k resamples), **uncorrected for multiple comparisons** → directional. Mind the per-cell n in §0 (~15/cell at 30 queries; run the full 72 to tighten).
- **Fan-out / retrieval judges** may be N/A for the single/generic variants — blank bars are by design.
- **Loads ONLY post-split data** from `RESULTS_DIR`; never mixes old/pre-split/relabeled runs. Figures export to `RESULTS_DIR/figures/` (PDF+PNG) on Run-All.